In [1]:
#Hits from my dissertation and PhD
#https://repositorio.ufpe.br/handle/123456789/5864
#https://repositorio.ufpe.br/handle/123456789/4950
# PACKAGE INSTALLATION AND LOADING
#%pip install pgmpy #Probabilistic Graphical Models with Python https://pgmpy.org/

In [2]:
# from pgmpy.model2s import BayesianNetwork
# %pip install pgmpy
from pgmpy.inference import VariableElimination
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.factors.discrete import TabularCPD
import networkx as nx
# import pylab as plt
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'pgmpy'

## BD01.csv

In [ ]:
import pandas as pd
bd01 = pd.read_csv(filepath_or_buffer = "../data/BD01.csv", 
                  sep=";")
display(bd01.head(3))

In [ ]:
# %pip install sklearn
import numpy as np
import sklearn
from sklearn.preprocessing import KBinsDiscretizer
disc = KBinsDiscretizer(n_bins=2, encode="ordinal", strategy='quantile')
qtQl = disc.fit_transform(bd01[['TempoFalha', 'nReincidenciaFalhas', 'CustoManutencao']])
# Accessing the interval edges for each feature:
print('disc.bin_edges_:', disc.bin_edges_)

qtQl = pd.DataFrame(qtQl, 
                            columns=['q_TempoFalha', 'q_nReincidenciaFalhas', 'q_CustoManutencao'],
                            dtype=np.int8)
display(qtQl)

In [ ]:
bd01 = pd.concat([bd01, qtQl], axis=1)
display(bd01)

In [ ]:
model_bd01 = DiscreteBayesianNetwork([('Fornecedor', 'Degradacao'),
                          ('Degradacao', 'q_TempoFalha'),
                          ('Degradacao', 'q_nReincidenciaFalhas'),
                          ('Degradacao', 'q_CustoManutencao')])  # arcs from the cause to the effect
model_daft = model_bd01.to_daft(node_pos = 'shell', latex=False)
# To open the plot
model_daft.render();

In [ ]:
# LEARNING PARAMETERS FROM DATA
# from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.estimators import BayesianEstimator
# Similarly to sklearn you can fit all CPDs of `model` using MLE:
model_bd01.fit(bd01, estimator=BayesianEstimator)
print(model_bd01.get_cpds('Fornecedor'))  # unconditional
print(model_bd01.get_cpds('Degradacao'))  # conditional
print(model_bd01.get_cpds('q_CustoManutencao'))  # conditional
# print(model2.get_cpds('correctAnswer_Q01'))  # conditional
# print(be_model2.get_cpds('correctAnswer_Q01'))  # conditional
# print(model2.get_cpds('correctAnswer_Q02'))  # conditional


In [ ]:
infer = VariableElimination(model_bd01)
posterior_p2 = infer.query(['q_CustoManutencao'], 
                           evidence={'Fornecedor': 'A'})
print(posterior_p2)

In [ ]:
posterior_p2 = infer.query(['q_CustoManutencao'], evidence={'Fornecedor': 'B'})
print(posterior_p2)

In [ ]:
posterior_p3 = infer.query(['Fornecedor'], evidence={'q_CustoManutencao': 0})
print(posterior_p3)

In [ ]:
2978 - 2100